In [52]:
import sqlite3
from datetime import datetime
from urllib.request import Request, urlopen

import pandas as pd
from bs4 import BeautifulSoup

In [53]:
URL = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

JSON_PATH = "Countries_by_GDP.json"
DB_NAME = "World_Economies.db"
TABLE_NAME = "Countries_by_GDP"

# 로그 파일명은 etl_project_log.txt
LOG_FILE = "etl_project_log.txt"

In [54]:
# log_progress 함수 정의
# ETL 프로세스에 따라 코드를 작성하고 각 단계의 시작과 끝을 로그에 기록.

def log_progress(message):
    """ETL 작업 진행 상황을 로그 파일에 append 모드로 기록한다."""
    timestamp = datetime.now().strftime("%Y-%B-%d-%H-%M-%S")

    # 기존 파일에 append 모드로 로그를 기록
    with open(LOG_FILE, "a", encoding="utf-8") as log_file:
        # 로그는 "time, log" 형식
        log_file.write(f"{timestamp}, {message}\n")

In [55]:
log_progress("ETL Job Started")

In [56]:
with open(LOG_FILE, "r", encoding="utf-8") as log_file:
    print(log_file.read())

2026-July-03-11-39-19, ETL Job Started



In [57]:
# get_soup 함수 정의
# URL에서 HTML을 가져와 BeautifulSoup 객체로 변환한다.

def get_soup(url):
    request = Request(
        url,
        # 웹페이지 접속 요청을 조금 더 안정적으로 하기 위해 User-Agent를 설정
        headers={"User-Agent": "Mozilla/5.0"}
    )

    html = urlopen(request).read()

    # BeautifulSoup에 넣어서 분석 가능한 개체로 변환
    soup = BeautifulSoup(html, "html.parser")

    return soup

In [58]:
# ETL

log_progress("Extract phase Started")

soup = get_soup(URL)

print(type(soup))
print(soup.title.text)

<class 'bs4.BeautifulSoup'>
List of countries by GDP (nominal) - Wikipedia


In [59]:
# IMF GDP 데이터 중에서 나라와 GDP만 추출

# 1. 페이지 안의 모든 wikitable 찾기
tables = soup.find_all("table", {"class": "wikitable"})

# 2. IMF가 들어있는 GDP 메인 표 찾기
for table in tables:
    table_text = table.get_text(" ", strip=True)

    if "Country/Territory" in table_text and "IMF" in table_text:
        target_table = table
        break

# 3. 찾은 표 안에서 국가명과 IMF GDP만 추출
for row in target_table.find_all("tr"):
    cells = row.find_all(["td", "th"])

    country = cells[0].get_text(" ", strip=True)
    imf_gdp = cells[1].get_text(" ", strip=True)

    print(f"{country} | {imf_gdp}")

Country/Territory | IMF ( 2026 ) [ 1 ]
World | 126,295,331
United States | 32,383,920
China [ n 1 ] | 20,851,593
Germany | 5,452,858
Japan | 4,379,253
United Kingdom | 4,264,794
India | 4,153,191
France | 3,596,094
Italy | 2,738,164
Russia | 2,656,452
Brazil | 2,635,912
Canada | 2,507,340
Australia | 2,123,963
Mexico | 2,120,855
Spain | 2,091,222
South Korea | 1,931,008
Turkey | 1,640,223
Indonesia | 1,539,872
Netherlands | 1,449,704
Saudi Arabia | 1,388,676
Switzerland | 1,146,911
Poland | 1,134,248
Taiwan [ n 3 ] | 976,719
Ireland | 779,381
Belgium | 776,730
Sweden | 760,481
Israel | 719,848
Argentina | 688,378
Singapore | 659,572
Austria | 623,719
United Arab Emirates | 621,546
Norway | 599,406
Thailand | 579,996
Colombia | 539,530
Vietnam | 527,266
Malaysia | 516,428
Philippines | 512,222
Bangladesh | 510,705
Denmark | 503,772
Romania | 480,834
South Africa | 479,964
Pakistan | 452,192
Hong Kong [ n 4 ] | 450,138
Czech Republic | 432,597
Egypt | 429,645
Chile | 407,850
Peru | 380,9

In [60]:
# 데이터 프레임 구축

import re

# 빈리스트 만들기
data = []

for row in target_table.find_all("tr"):
    cells = row.find_all(["td", "th"])

    # 셀이 부족한 행은 건너뛰기
    if len(cells) < 2:
        continue

    country = cells[0].get_text(" ", strip=True)
    imf_gdp = cells[1].get_text(" ", strip=True)

    # 나라 이름 뒤의 notes 제거: 예) China [ n 1 ] -> China
    country = re.sub(r"\s*\[.*?\]", "", country).strip()
    # GDP 값 뒤 연도 제거: 98,964 (2024) -> 98,964
    imf_gdp = re.sub(r"\s*\(.*?\)", "", imf_gdp).strip()

    # 헤더나 불필요한 행 제거
    if country in ["Country/Territory", "World", ""] or imf_gdp == "":
        continue

    # dict 형태로 데이터를 리스트에 추가
    data.append({
        "Country": country,
        "GDP_USD_million": imf_gdp
    })

raw_gdp_df = pd.DataFrame(data)

raw_gdp_df.head()

,Country,GDP_USD_million
0,United States,"32,383,920"
1,China,"20,851,593"
2,Germany,"5,452,858"
3,Japan,"4,379,253"
4,United Kingdom,"4,264,794"


In [61]:
gdp_df = raw_gdp_df.copy()

# GDP 값에서 쉼표, 각주, 문자 등을 제거
gdp_df["GDP_USD_million"] = (
    gdp_df["GDP_USD_million"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace(r"[^0-9.]", "", regex=True)
)

# 숫자형으로 변환
gdp_df["GDP_USD_million"] = pd.to_numeric(
    gdp_df["GDP_USD_million"],
    errors="coerce"
)

# GDP 값이 없는 행 제거
gdp_df = gdp_df.dropna(subset=["GDP_USD_million"])

# million USD → billion USD 변환
gdp_df["GDP_USD_billion"] = (
    gdp_df["GDP_USD_million"] / 1000
).round(2)

# 필요한 컬럼만 선택
gdp_df = gdp_df[["Country", "GDP_USD_billion"]]

# GDP 높은 순서로 정렬
gdp_df = gdp_df.sort_values(
    by="GDP_USD_billion",
    ascending=False
).reset_index(drop=True)

gdp_df.head(10)

,Country,GDP_USD_billion
0,United States,32383.92
1,China,20851.59
2,Germany,5452.86
3,Japan,4379.25
4,United Kingdom,4264.79
5,India,4153.19
6,France,3596.09
7,Italy,2738.16
8,Russia,2656.45
9,Brazil,2635.91


In [62]:
gdp_over_100b = gdp_df[gdp_df["GDP_USD_billion"] >= 100]

print("GDP가 100B USD 이상인 국가")
display(gdp_over_100b)

GDP가 100B USD 이상인 국가


,Country,GDP_USD_billion
0,United States,32383.92
1,China,20851.59
2,Germany,5452.86
3,Japan,4379.25
4,United Kingdom,4264.79
...,...,...
76,Venezuela,111.30
77,Luxembourg,110.42
78,Costa Rica,109.93
79,Lithuania,105.91


In [63]:
# country_converter 라이브러리를 사용
import country_converter as coco
# country_converter는 국가명을 표준화하거나,
# 국가명을 기준으로 대륙(continent), ISO 코드 등을 변환해주는 외부 라이브러리이다.
# 이번 과제의 GDP 원본 테이블에는 Region 컬럼이 없기 때문에,
# 국가명(Country)을 기준으로 Region 정보를 추가하기 위해 사용한다.

cc = coco.CountryConverter()

# 국가명을 대륙명으로 변환하여 새로운 컬럼 "Region"에 추가
gdp_df["Region"] = cc.convert(
    names=gdp_df["Country"],
    to="continent",
    not_found="Other"
)


gdp_df = gdp_df[["Country", "Region", "GDP_USD_billion"]]

gdp_df.head(10)

,Country,Region,GDP_USD_billion
0,United States,America,32383.92
1,China,Asia,20851.59
2,Germany,Europe,5452.86
3,Japan,Asia,4379.25
4,United Kingdom,Europe,4264.79
5,India,Asia,4153.19
6,France,Europe,3596.09
7,Italy,Europe,2738.16
8,Russia,Europe,2656.45
9,Brazil,America,2635.91


In [64]:
# GDP가 100B USD 이상인 국가만 필터링
gdp_over_100b = gdp_df[gdp_df["GDP_USD_billion"] >= 100]

print("GDP가 100B USD 이상인 국가")
display(gdp_over_100b)

GDP가 100B USD 이상인 국가


,Country,Region,GDP_USD_billion
0,United States,America,32383.92
1,China,Asia,20851.59
2,Germany,Europe,5452.86
3,Japan,Asia,4379.25
4,United Kingdom,Europe,4264.79
...,...,...,...
76,Venezuela,America,111.30
77,Luxembourg,Europe,110.42
78,Costa Rica,America,109.93
79,Lithuania,Europe,105.91


In [65]:
# 각 Region별 GDP Top5 국가의 GDP 평균

region_top5 = (
    gdp_df
    .sort_values(["Region", "GDP_USD_billion"], ascending=[True, False])
    .groupby("Region")
    .head(5)
)

region_top5_avg = (
    region_top5
    .groupby("Region")["GDP_USD_billion"]
    .mean()
    .round(2)
    .reset_index()
)

region_top5_avg.columns = ["Region", "Top5_Avg_GDP_USD_billion"]

print("각 Region별 GDP Top5 국가의 GDP 평균")
display(region_top5_avg)

각 Region별 GDP Top5 국가의 GDP 평균


,Region,Top5_Avg_GDP_USD_billion
0,Africa,359.69
1,America,8067.28
2,Asia,6591.05
3,Europe,3741.67
4,Oceania,489.04


In [66]:
# JSON 파일로 저장
raw_gdp_df.to_json(
    JSON_PATH,
    orient="records",
    indent=4,
    force_ascii=False
)

In [67]:
import os

print(os.path.exists(JSON_PATH))

True


In [68]:
# 가공된 GDP DataFrame을 SQLite 데이터베이스에 저장

def load_to_database(df, db_name, table_name):
    log_progress("Load phase Started")

    conn = sqlite3.connect(db_name)

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    log_progress("Load phase Ended")

In [69]:
load_to_database(gdp_df, DB_NAME, TABLE_NAME)

In [70]:
import os

print(os.path.exists(DB_NAME))

True


In [71]:
conn = sqlite3.connect(DB_NAME)

query = """
SELECT *
FROM Countries_by_GDP
ORDER BY GDP_USD_billion DESC
LIMIT 10;
"""

sql_result = pd.read_sql(query, conn)

conn.close()

sql_result

,Country,Region,GDP_USD_billion
0,United States,America,32383.92
1,China,Asia,20851.59
2,Germany,Europe,5452.86
3,Japan,Asia,4379.25
4,United Kingdom,Europe,4264.79
5,India,Asia,4153.19
6,France,Europe,3596.09
7,Italy,Europe,2738.16
8,Russia,Europe,2656.45
9,Brazil,America,2635.91


# Ai 사용처
chatGPT를 사용하여 요구상황에 맞는 함수이름과 함수용법을 검색했다.